In [0]:
CREATE OR REPLACE VIEW workspace.default.gold_audit_trail AS
SELECT
    ROW_NUMBER() OVER (ORDER BY step) AS transaction_id,
    event_time,
    type AS transaction_type,
    amount,
    nameOrig AS sender_account,
    nameDest AS receiver_account,
    isFraud AS ground_truth_fraud,
    orig_drained_to_zero AS flag_account_drained,
    CASE
        WHEN isFraud = 1 THEN 'CONFIRMED_FRAUD'
        WHEN orig_drained_to_zero = 1 AND amount > 10000 THEN 'HIGH_RISK_REVIEW'
        ELSE 'NORMAL'
    END AS risk_category
FROM workspace.default.feature_transactions;
SELECT risk_category, COUNT(*) AS count
FROM workspace.default.gold_audit_trail
GROUP BY risk_category
ORDER BY count DESC;